In [1]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.express as px
import os
import sys
import datetime as dt

In [2]:

sys.path.insert(1, '..')
from SepsisPkg.config import SEPSIS_CONFIG

In [3]:
# Constants
RAW_DATA_PATH = "../data/raw_data_new"
files = sorted(os.listdir(RAW_DATA_PATH))

In [4]:
def cast_datetime_cols(
	lf: pl.LazyFrame,
	cols = ['Pt_Arrival', 'Event_DateTime', 'Admit_Time']
):
	schema = lf.collect_schema()
	for c in cols:
		if schema[c] in [pl.Utf8, pl.String]:
			lf = lf.with_columns(
			pl.col(c).str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S%.f")
		)
	return lf

In [5]:
lf_list = []
meas_value_char = []
meas_value_rate = []

lf_baseline = None
for idx, f in enumerate(files):
    if f == 'SOFA Scores - 1.13.26.csv': continue
    f_path = os.path.join(RAW_DATA_PATH, f)
    lf_f = pl.scan_csv(
        f_path,
        null_values=['null', 'NULL', 'Null', "None"],
        infer_schema_length=int(1e7)
    )

    schema = lf_f.collect_schema()
    if 'Event_Datetime' in schema.names():
        print("renamed")
        lf_f = lf_f.rename(
            {'Event_Datetime': 'Event_DateTime'}
        )
    if 'MEAS_VALUE_RATE' in schema.names():
        meas_value_rate.append((lf_f.filter(
            pl.col("MEAS_VALUE_RATE").is_not_null()
            # pl.col("MEAS_VALUE_RATE").is_not_nan()
        ).select("PAT_ENC_CSN_ID", "Event_DateTime", "EVENT_NAME", "MEAS_VALUE_RATE").collect(), f))
        lf_f = lf_f.drop("MEAS_VALUE_RATE")

    if 'MEAS_VALUE_CHAR' in schema.names():
        meas_value_char.append((lf_f.filter(
            pl.col("MEAS_VALUE_CHAR").is_not_null()
            # pl.col("MEAS_VALUE_CHAR").is_not_nan()
        ).select("PAT_ENC_CSN_ID", "Event_DateTime", "EVENT_NAME", "MEAS_VALUE_CHAR").collect(), f))
        lf_f = lf_f.drop("MEAS_VALUE_CHAR")
    
    if  'MEAS_VALUE' in schema.names() and schema['MEAS_VALUE'] not in [pl.Utf8, pl.String]:
        lf_f = lf_f.with_columns(
           pl.col("MEAS_VALUE").cast(pl.Utf8)
        )
    if 'Before_Admit_YN' in schema.names():
        print(f"Dropping Before_Admit_YN for file {f}\n")
        lf_f = lf_f.drop("Before_Admit_YN")

    if 'Event_ID' in schema.names():
        print(f"Dropping Event_ID for file {f}\n")
        lf_f = lf_f.drop("Event_ID")
    
    if f == 'Encounter Table with Baseline Values - Dec 2024 - Nov 2025.csv':
        lf_baseline = lf_f
        continue
    
    lf_list.append(lf_f)

renamed
Dropping Before_Admit_YN for file Flowsheet Character and BP - Dec 2024 - Nov 2025.csv

Dropping Event_ID for file Flowsheet Character and BP - Dec 2024 - Nov 2025.csv

renamed
Dropping Before_Admit_YN for file Flowsheet Value - Dec 2024 - Nov 2025.csv

Dropping Event_ID for file Flowsheet Value - Dec 2024 - Nov 2025.csv

Dropping Before_Admit_YN for file LVEF, Proc Orders, Dx - Dec 2024 - Nov 2025.csv

Dropping Event_ID for file LVEF, Proc Orders, Dx - Dec 2024 - Nov 2025.csv

Dropping Before_Admit_YN for file Lab Results - Dec 2024 - Nov 2025.csv

Dropping Event_ID for file Lab Results - Dec 2024 - Nov 2025.csv

Dropping Before_Admit_YN for file Med Orders and Admin - Dec 2024 - Nov 2025.csv

Dropping Event_ID for file Med Orders and Admin - Dec 2024 - Nov 2025.csv



In [6]:
lf_all = pl.concat(
    lf_list
)

In [7]:
lf_all = cast_datetime_cols(lf_all, ['Pt_Arrival', 'Event_DateTime', 'Admit_Time'])
lf_all = lf_all.with_columns(
    event_full_name = (pl.col("Type")+"_"+pl.col("EVENT_NAME")+"_"+pl.col("Event_Grouper")).str.to_lowercase()
).sort(by=['PAT_ENC_CSN_ID', 'Event_DateTime'])

In [8]:
included_cultures = [
    # Blood cultures
    "CULTURE BLOOD",
    "CULTURE BLOOD ID MALDI - AEROBIC",
    "CULTURE BLOOD ID MALDI - ANAEROBIC",
    "CULTURE BLOOD ID MALDI-AEROBIC",
    "CULTURE BLOOD ID MALDI-ANAEROBIC",
    "CULTURE BLOOD ID NAAT- AEROBIC",
    "CULTURE BLOOD ID NAAT- ANAEROBIC",

    # Fungal and mycobacterial blood cultures
    "FUNGAL BLOOD CULTURE",
    "FUNGAL CULTURE, BLOOD",
    "MYCOBACTERIAL CULTURE, BLOOD",
    "AFB BLOOD CULTURE",

    # Sterile site cultures
    "CULTURE STERILE SITE",
    "CULTURE CSF + GRAM STAIN",
    "CULTURE ANAEROBIC",
    "CULTURE AEROBIC + ANAEROBIC + GRAM STAIN",
    "CULTURE AEROBIC AND ANAEROBIC RESULT",

    # Device-related sterile cultures
    "CULTURE CATH TIP",
    "CULTURE URINE",
    "CULTURE SPUTUM + SCREENING SMEAR",
    "CULTURE RESPIRATORY (NON SPUTUM) + STAIN",
]

excluded_cultures = [
    # BCID organism identifications (results, not orders)
    "COAGULASE-NEGATIVE STAPHYLOCOCCUS BCID",
    "KLEBSIELLA AEROGENES GROUP BCID",
    "KLEBSIELLA PNEUMONIAE GROUP BCID",
    "KLEBSIELLA OXYTOCA BCID",
    "ESCHERICHIA COLI BCID",
    "SERRATIA MARCESCENS BCID",
    "ACINETOBACTER CALCOACETICUS-BAUMANNII COMPLEX BCID",
    "STENOTROPHOMONAS MALTOPHILIA BCID",
    "ENTEROBACTER CLOACAE COMPLEX BCID",
    "STAPHYLOCOCCUS SP BCID",
    "STAPHYLOCOCCUS LUGDUNENSIS BCID",
    "STAPHYLOCOCCUS AUREUS BCID",
    "STREPTOCOCCUS SP BCID",
    "STREPTOCOCCUS PNEUMONIAE BCID",
    "GROUP B STREPTOCOCCUS BCID",
    "ENTEROCOCCUS FAECALIS BCID",
    "ENTEROCOCCUS FAECIUM BCID",
    "PROTEUS SP BCID",
    "HAEMOPHILUS INFLUENZAE BCID",
    "PSEUDOMONAS AERUGINOSA BCID",
    "SALMONELLA SP BCID",
    "BACTEROIDES FRAGILIS BCID",
    "LISTERIA MONOCYTOGENES BCID",
    "ENTEROBACTERALES",

    # Resistance and gene markers
    "CTX-M GENE BCID",
    "OXA GENE BCID",
    "NDM GENE",
    "VANAB GENE BCID",
    "MEC A/C AND MREJ (MRSA) GENE",

    # Fungal or organism species results
    "CANDIDA ALBICANS",
    "CANDIDA KRUSEI",
    "CANDIDA PARAPSILOSIS",
    "CANDIDA TROPICALIS",
    "NAKASEOMYCES CANDIDA GLABRATA",
    "CANDIDA AURIS",
    "CRYPTOCOCCUS NEOFORMANS/GATTII",

    # Stains, smears, and microscopy
    "GRAM STAIN",
    "GRAM ST",
    "AFB SMEAR",
    "MODIFIED ACID FAST STAIN",
    "FUNGAL SMEAR",

    # Status, metadata, and generic rows
    "BCID NAAT INTERPRETATION",
    "CULTURE STATUS",
    "RESULT",
    "ISOLATE 1",
    "CULTURE SOURCE",
    "CULTURE",
]

more_refined_included_cultures = [
    # Primary 
    "CULTURE BLOOD",
    "FUNGAL BLOOD CULTURE",
    "FUNGAL CULTURE, BLOOD",
    "MYCOBACTERIAL CULTURE, BLOOD",
    "AFB BLOOD CULTURE",
    "CULTURE STERILE SITE",
    "CULTURE CSF + GRAM STAIN",
    "CULTURE ANAEROBIC",
    "CULTURE AEROBIC + ANAEROBIC + GRAM STAIN",
    "CULTURE AEROBIC AND ANAEROBIC RESULT",
    "CULTURE CATH TIP",

    # Secondary
    "CULTURE URINE",
    "CULTURE SPUTUM + SCREENING SMEAR",
    "CULTURE RESPIRATORY (NON SPUTUM) + STAIN",
]

included_cultures_lowercase = list(map(lambda x: x.lower(), included_cultures))

def get_AI_based_filter():
    expr_antibiotic_filter = (
    (pl.col("Event_Grouper") == 'Antibiotics')&
    ~(pl.col("EVENT_NAME").str.to_lowercase().str.contains(r"topical|oral|otic|lock|intravitreal|irrigation"))
    )
    expr_culture_filter = (
        pl.col("event_full_name").str.ends_with("body fluid cultures") & 
        pl.col("event_full_name").str.contains(fr"{'|'.join(included_cultures_lowercase)}") 
        
    )
    return expr_antibiotic_filter, expr_culture_filter

def get_all_antibiotic_cultures():
    expr_antibiotic_filter = (
        pl.col("event_full_name").str.ends_with("antibiotics") 
    )
    expr_culture_filter = (
        pl.col("event_full_name").str.ends_with("body fluid cultures") 
    )
    return expr_antibiotic_filter, expr_culture_filter

def get_expressions(with_AI_filter=True):
    if with_AI_filter:
        return get_AI_based_filter()
    else:
        return get_all_antibiotic_cultures()

In [9]:
# df_all = lf_all.collect()

In [10]:
expr_anti, expr_culture = get_expressions(False)
expr_anti_ai, expr_culture_ai = get_expressions(True)

In [56]:
from typing import Union

def get_sepsis_suspected(df_all: Union[pl.DataFrame, pl.LazyFrame], expr_anti: pl.Expr, expr_culture: pl.Expr):
    df_antibiotic_all = df_all.filter(
        expr_anti
    ).select(
        "PAT_ENC_CSN_ID",
        pl.col("Event_DateTime").alias("ABX_time"),
        pl.col("event_full_name").alias("ABX")
    )

    df_culture_all = df_all.filter(
    expr_culture 
    ).select(
        "PAT_ENC_CSN_ID",
        pl.col("Event_DateTime").alias("culture_time"),
        pl.col("event_full_name").alias("culture_name")
    )

    df_suspected_backward = df_antibiotic_all.join_asof(
        df_culture_all,
        left_on='ABX_time',
        right_on='culture_time',
        strategy="backward",
        by="PAT_ENC_CSN_ID",
        tolerance="72h"
    ).filter(pl.col("culture_time").is_not_null())

    df_suspected_forward = df_antibiotic_all.join_asof(
        df_culture_all,
        left_on='ABX_time',
        right_on='culture_time',
        strategy="forward",
        by="PAT_ENC_CSN_ID",
        tolerance="24h"
    ).filter(pl.col("culture_time").is_not_null())

    df_f_b = pl.concat([df_suspected_backward, df_suspected_forward])
    return df_f_b

lf_f_b = get_sepsis_suspected(lf_all, expr_anti, expr_culture)
lf_f_b_ai = get_sepsis_suspected(lf_all, expr_anti_ai, expr_culture_ai)

In [12]:
lf_sofa = pl.scan_csv(
    os.path.join(RAW_DATA_PATH, "SOFA Scores - 1.13.26.csv"),
    null_values=["NULL", "Null", 'null', 'none'],
    infer_schema_length=int(1e6)
)
lf_sofa = cast_datetime_cols(lf_sofa, ["Score_Start", "Score_End"])

In [27]:
lf_joined = lf_all.join_asof(
    lf_sofa.select("EncounterEpicCsn", "Score_Start",  "Score_End", "SOFA").rename({"EncounterEpicCsn":"PAT_ENC_CSN_ID"}),
    right_on="Score_Start",
    left_on="Event_DateTime",
    by="PAT_ENC_CSN_ID",
    strategy="backward"
).filter(
    pl.col("Score_End").is_null() | (pl.col("Event_DateTime") <= pl.col("Score_End"))
)

In [28]:
future_asof = lf_joined.collect_async()

sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


In [29]:
if future_asof.result.done():
    df_joined_asof = future_asof.result.result()
else: print("Circle back")

In [33]:
df_joined_asof = df_joined_asof.with_columns(
    pl.col("SOFA").fill_null(0)
)

In [57]:
df_f_b =lf_f_b.collect()
df_f_b_ai =lf_f_b_ai.collect()

sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


In [37]:
df_f_b

PAT_ENC_CSN_ID,ABX_time,ABX,culture_time,culture_name
i64,datetime[μs],str,datetime[μs],str
695960811,2024-04-12 08:58:09,"""medication - administration_le…",2024-04-12 06:51:13,"""order - procedure_culture bloo…"
695960811,2024-04-12 09:11:16,"""medication - administration_pi…",2024-04-12 06:51:13,"""order - procedure_culture bloo…"
695960811,2024-04-12 18:15:13,"""medication - administration_pi…",2024-04-12 06:51:13,"""order - procedure_culture bloo…"
695960811,2024-04-13 01:56:48,"""medication - administration_pi…",2024-04-12 06:51:13,"""order - procedure_culture bloo…"
695960811,2024-04-15 09:16:27,"""medication - administration_le…",2024-04-14 23:02:19,"""order - procedure_culture bloo…"
…,…,…,…,…
741451421,2025-11-29 23:30:09,"""medication - administration_me…",2025-11-30 07:45:00,"""order result - microbiology_cu…"
741451421,2025-11-30 10:01:57,"""medication - administration_ce…",2025-12-01 06:51:00,"""order result - microbiology_cu…"
741451421,2025-11-30 10:08:54,"""medication - administration_me…",2025-12-01 06:51:00,"""order result - microbiology_cu…"


In [58]:
# Join with AI filtered antibiotics and culture
df_joined_abx_backward = df_joined_asof.join_asof(
    df_f_b_ai,
    left_on="Event_DateTime",
    right_on="ABX_time",
    by="PAT_ENC_CSN_ID",
    strategy="backward",
    tolerance="48h"
)

df_joined_abx_forward = df_joined_asof.join_asof(
    df_f_b_ai,
    left_on="Event_DateTime",
    right_on="ABX_time",
    by="PAT_ENC_CSN_ID",
    strategy="forward",
    tolerance="48h"
)

df_joined_abx_f_b_ai = pl.concat([df_joined_abx_backward, df_joined_abx_forward])

/tmp/ipykernel_1689680/3293278954.py:2: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_joined_abx_backward = df_joined_asof.join_asof(
/tmp/ipykernel_1689680/3293278954.py:11: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_joined_abx_forward = df_joined_asof.join_asof(


In [38]:
df_joined_abx_backward = df_joined_asof.join_asof(
    df_f_b,
    left_on="Event_DateTime",
    right_on="ABX_time",
    by="PAT_ENC_CSN_ID",
    strategy="backward",
    tolerance="48h"
)

df_joined_abx_forward = df_joined_asof.join_asof(
    df_f_b,
    left_on="Event_DateTime",
    right_on="ABX_time",
    by="PAT_ENC_CSN_ID",
    strategy="forward",
    tolerance="48h"
)

df_joined_abx_f_b = pl.concat([df_joined_abx_backward, df_joined_abx_forward])

/tmp/ipykernel_1689680/1629960005.py:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_joined_abx_backward = df_joined_asof.join_asof(
/tmp/ipykernel_1689680/1629960005.py:10: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_joined_abx_forward = df_joined_asof.join_asof(


In [72]:
df_joined_abx_f_b = df_joined_abx_f_b.sort(by=['PAT_ENC_CSN_ID', 'Event_DateTime']).filter(pl.col("ABX_time").is_not_null())
df_joined_abx_f_b = df_joined_abx_f_b.with_columns(
    admit_2_abx_hours = (pl.col("Event_DateTime")-pl.col("Admit_Time")).dt.total_hours()
)
# df_joined_abx_f_b.filter(pl.col('admit_2_abx_hours')<12).select("PAT_ENC_CSN_ID", "Sepsis_Category", "SOFA", "Event_DateTime", "ABX", "ABX_time", "admit_2_abx_hours")
# df_joined_abx_f_b.filter((pl.col('SOFA')>=2)&(pl.col("admit_2_abx_hours")<=48)).select("PAT_ENC_CSN_ID", "Sepsis_Category", "SOFA", "Event_DateTime", "ABX", "ABX_time", "admit_2_abx_hours")
df_joined_abx_f_b.filter((pl.col('SOFA')>=2)&(pl.col("admit_2_abx_hours")<=48)&(pl.col("Sepsis_Category").str.starts_with("NPOA"))).select("PAT_ENC_CSN_ID", "Sepsis_Category", "SOFA", "Event_DateTime", "ABX", "ABX_time", "admit_2_abx_hours")['PAT_ENC_CSN_ID'].n_unique()

296

In [73]:
df_joined_abx_f_b_ai = df_joined_abx_f_b_ai.sort(by=['PAT_ENC_CSN_ID', 'Event_DateTime']).filter(pl.col("ABX_time").is_not_null())
df_joined_abx_f_b_ai = df_joined_abx_f_b_ai.with_columns(
    admit_2_abx_hours = (pl.col("Event_DateTime")-pl.col("Admit_Time")).dt.total_hours()
)
# df_joined_abx_f_b_ai.filter(pl.col('admit_2_abx_hours')<12).select("PAT_ENC_CSN_ID", "Sepsis_Category", "SOFA", "Event_DateTime", "ABX", "ABX_time", "admit_2_abx_hours")
df_joined_abx_f_b_ai.filter((pl.col('SOFA')>=2)&(pl.col("admit_2_abx_hours")<=48)).select("PAT_ENC_CSN_ID", "Sepsis_Category", "SOFA", "Event_DateTime", "ABX", "ABX_time", "admit_2_abx_hours")
df_joined_abx_f_b_ai.filter((pl.col('SOFA')>=2)&(pl.col("admit_2_abx_hours")<=48)&(pl.col("Sepsis_Category").str.starts_with("NPOA"))).select("PAT_ENC_CSN_ID", "Sepsis_Category", "SOFA", "Event_DateTime", "ABX", "ABX_time", "admit_2_abx_hours")['PAT_ENC_CSN_ID'].n_unique()

286

In [79]:
df_joined_abx_f_b_ai.filter((pl.col('SOFA')>=2)&(pl.col("admit_2_abx_hours")<=48)&(pl.col("Sepsis_Category").str.starts_with("NPOA"))).select("PAT_ENC_CSN_ID", "Sepsis_Category", "SOFA", "Event_DateTime", "ABX", "ABX_time", "admit_2_abx_hours").unique().sort(by=['PAT_ENC_CSN_ID', 'Event_DateTime']).write_csv("missed_NPOA_within_48hrs.csv")

In [94]:
df_joined_abx_f_b_ai.filter(
    pl.col("PAT_ENC_CSN_ID").is_in(df_joined_abx_f_b_ai.group_by("PAT_ENC_CSN_ID").agg(
    pl.col("admit_2_abx_hours").first()
).filter( (pl.col("admit_2_abx_hours")>=24*7) )['PAT_ENC_CSN_ID'])).filter(pl.col("Sepsis_Category").str.starts_with("NPOA-1"))['PAT_ENC_CSN_ID'].unique().to_list()

/tmp/ipykernel_1689680/3395551570.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_joined_abx_f_b_ai.filter(


[713257513,
 713608612,
 714445951,
 714904379,
 715249765,
 715653135,
 715879406,
 716118582,
 717354323,
 718886311,
 719226357,
 719807256,
 720849657,
 721039602,
 722144861,
 724949806,
 724954844,
 725126078,
 725428557,
 725720478,
 726856225,
 726968521,
 727144441,
 728086896,
 728572563,
 728659920,
 729013534,
 729213163,
 729952543,
 731075983,
 731386156,
 732723600,
 732887839,
 733062372,
 733066002,
 734030192,
 734256396,
 734537262,
 734684129,
 738338345,
 738640673,
 738906216]